# Compare the two halves of a video

Explore whether the first and second halves look similar. These scores are not yet selection rules.

From this directory: `uv sync`, then `uv run jupyter lab`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display
from dbp_metrics.experimental.half_similarity import Config, decode_media, histogram, cosine, measure

demo = Path("data/nsd_subj01.mp4")
VIDEO_PATH = demo if demo.exists() else Path("notebooks/data/nsd_subj01.mp4")
config = Config()
display(Video(str(VIDEO_PATH), embed=True))

## 1. Split and sample

Split at half the duration. Each half requests 64 evenly spaced times and uses nearby decoded frames within that half. Short halves may reuse frames. The comparison uses 256×256 center crops, not the full image.

In [ ]:
decoded = decode_media(VIDEO_PATH, config)
print("Duration:", decoded["duration_seconds"], "seconds")
print("Split:", decoded["split_seconds"], "seconds")
for index, half in enumerate(decoded["halves"], start=1):
    print(f"Half {index}: {len(half['frames'])} samples, "
          f"{len(set(half['frame_indices']))} distinct frames")

In [ ]:
figure, axes = plt.subplots(2, 3, figsize=(10, 6))
for row, half in enumerate(decoded["halves"]):
    for column, sample in enumerate([0, len(half["frames"]) // 2, len(half["frames"]) - 1]):
        frame = np.frombuffer(half["frames"][sample], dtype=np.uint8).reshape(256, 256, 3)
        axes[row, column].imshow(frame)
        axes[row, column].set_title(f"Half {row + 1}: {half['timestamps_seconds'][sample]:.2f}s")
        axes[row, column].axis("off")
plt.tight_layout()

## 2. Visual baseline

Count red, green and blue pixel intensities in each half. Compare the resulting vectors:

$$s=\\frac{a\\cdot b}{\\|a\\|\\|b\\|}$$

Similar colors can produce a high score even when the scene or action differs. This baseline does not measure meaning.

In [ ]:
vectors = [histogram(half["frames"], config.bins) for half in decoded["halves"]]
visual_score = cosine(*vectors)
print("Visual histogram cosine:", visual_score)
figure, axis = plt.subplots(figsize=(10, 3))
for index, vector in enumerate(vectors, start=1):
    axis.plot(vector, label=f"Half {index}")
axis.set_xlabel("Concatenated RGB histogram bins")
axis.set_ylabel("Histogram value")
axis.legend()
plt.show()

## 3. Optional V-JEPA comparison

V-JEPA represents each half with a learned video embedding. Compare the embeddings using cosine similarity. This still needs manual evaluation; a high score is not a probability.

Enable only in a compatible environment with the pinned model already cached. Nothing is downloaded. See `requirements-similarity.txt` in the repository root.

In [ ]:
RUN_VJEPA = False
DEVICE = "cpu"

if RUN_VJEPA:
    from dbp_metrics.experimental.half_similarity import VJEPABackend
    backend = VJEPABackend(DEVICE)
    model_config = Config(vjepa=True, device=DEVICE)
    result = measure(VIDEO_PATH, model_config, backend=backend)
    print(json.dumps(result, indent=2))
else:
    print("V-JEPA not run. Visual baseline only.")

## 4. Manual comparison

Try clips with: a continuous action; the same scene but a different action; a scene cut; similar colors but different content. Record your judgment before viewing the scores. Do the scores agree with your ordering?

In [ ]:
comparisons = [
    {
        "video": str(VIDEO_PATH),
        "manual_similarity": None,
        "visual_histogram_cosine": visual_score,
        "vjepa_cosine": None,
        "notes": "",
    }
]
comparisons

## Decision

Compare both methods on a small, varied set. Inspect disagreements. Decide which measure answers the intended question before choosing a threshold or adding a website metric.